# Dedicated Per-Drug LR + MLP on Aggregated A+B+C+D

**10 independent models per drug type** on the aggregated 4-site dataset.

| Model | Details |
|---|---|
| **LR** | L2 Logistic Regression, GridSearchCV(C), CV threshold tuning |
| **MLP** | 8×8 grid lr×dropout, internal val + early stopping (patience=10) |

**Split:** Species-stratified 70/15/15 on pooled A+B+C+D per drug.
**Preprocessing:** log1p + standardise (fit on train only).
**Output:** Heatmaps -- models × drugs (Balanced Accuracy + AUC-ROC).

Compatible with Google Colab.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, train_test_split, cross_val_predict)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE_NAME}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_dedicated_lr_mlp")
OUT_DIR.mkdir(exist_ok=True)

DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]

# Tuning grids
C_GRID = np.linspace(5e-5, 1e-3, 15)
LR_GRID = np.linspace(7.5e-5, 9.5e-5, 8)
DROP_GRID = np.linspace(0.5, 0.8, 8)
THRESHOLDS = np.linspace(0.05, 0.95, 91)

print(f"Drugs: {len(DRUGS_10)}  |  C grid: {len(C_GRID)}  |  MLP grid: {len(LR_GRID)}x{len(DROP_GRID)}={len(LR_GRID)*len(DROP_GRID)}")

In [ ]:
# ── 1. LOAD AGGREGATED PER-DRUG DATA ──

def load_drug_data(drug_name):
    """Load a single drug's data from all 4 sites, concatenated."""
    frames = []
    dir_name = drug_name.replace(" ", "_")
    for site in SITES:
        p = DRYAD / "Processed" / f"Proc_{site}" / dir_name / "data.csv"
        if not p.exists(): continue
        df = pd.read_csv(p)
        df["site"] = site
        frames.append(df)
    df_all = pd.concat(frames, ignore_index=True)
    bin_cols = [c for c in df_all.columns if c.startswith("bin_")]
    X = df_all[bin_cols].to_numpy(dtype="float32")
    y = df_all["label"].to_numpy(dtype="int64")
    species = df_all["species"].to_numpy()
    return X, y, species

print("Checking data availability...")
for drug in DRUGS_10:
    X, y, _ = load_drug_data(drug)
    n_r = (y == 1).sum(); n_s = (y == 0).sum()
    print(f"  {drug:35s}  {len(y):>6d} samples  ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R)")
print()

In [ ]:
# ── 2. PYTORCH HELPERS ──

class BinDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def build_mlp(dropout_high=0.4):
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2,
        hidden_dim=512, head_dims=(256, 128),
        dropout_high=dropout_high, dropout_low=dropout_high / 2.0,
        use_attention=False)

In [ ]:
def predict_proba(model, X_np):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(next(model.parameters()).device)
    with torch.no_grad():
        return torch.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. MODEL FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def train_eval_lr(X_train, y_train, test_sets):
    """GridSearchCV(C) + CV threshold tuning. Returns {test_name: {BalAcc, AUC}}."""
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_c = grid.best_params_["C"]

    cv_proba = cross_val_predict(
        LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]

    lr = LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                            class_weight="balanced", max_iter=5000, random_state=SEED)
    lr.fit(X_train, y_train)

    results = {}
    for name, (X_te, y_te) in test_sets.items():
        proba = lr.predict_proba(X_te)[:, 1]
        preds = proba >= best_t
        results[name] = {
            "BalAcc": balanced_accuracy_score(y_te, preds),
            "AUC": roc_auc_score(y_te, proba),
            "Threshold": best_t, "Best_Param": f"C={best_c:.2e}",
        }
    return results

In [ ]:
def train_eval_mlp(X_train, y_train, test_sets):
    """8x8 grid lr x dropout, scored on external val set.
    Matches 04-02 baseline: no class weights, patience=10."""

    # Internal val fraction for early stopping (10% of train)
    X_ft, X_fv, y_ft, y_fv = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=SEED)

    best_ba, best_lr, best_dh = -1.0, None, None
    best_combo_model = None

    for lr_val in LR_GRID:
        for d in DROP_GRID:
            dh, dl = d, d / 2.0
            model = build_mlp(dh).to(DEVICE_NAME)
            train_dl = DataLoader(BinDataset(X_ft, y_ft), batch_size=64, shuffle=True)
            val_dl   = DataLoader(BinDataset(X_fv, y_fv), batch_size=128, shuffle=False)
            opt = torch.optim.AdamW(model.parameters(), lr=lr_val, weight_decay=1e-3)
            crit = nn.CrossEntropyLoss()
            best_vl, best_sd, patience = float("inf"), None, 0
            for ep in range(50):
                model.train()
                for xb, yb in train_dl:
                    xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                    opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
                model.eval(); vl = 0.0
                with torch.no_grad():
                    for xb, yb in val_dl:
                        xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                        vl += crit(model(xb), yb).item()
                vl /= len(val_dl)
                if vl < best_vl:
                    best_vl = vl; best_sd = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience = 0
                else:
                    patience += 1
                    if patience >= 10: break
            if best_sd is not None:
                model.load_state_dict(best_sd)
            # Score on EXTERNAL val set
            proba = predict_proba(model, test_sets["Val"][0])
            ba = balanced_accuracy_score(test_sets["Val"][1], proba >= 0.5)
            if ba > best_ba:
                best_ba = ba; best_lr = lr_val; best_dh = dh
                best_combo_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Load best combo weights into a fresh model, then retrain on FULL train
    model_final = build_mlp(best_dh).to(DEVICE_NAME)
    if best_combo_model is not None:
        model_final.load_state_dict(best_combo_model)
    train_dl = DataLoader(BinDataset(X_train, y_train), batch_size=64, shuffle=True)
    opt = torch.optim.AdamW(model_final.parameters(), lr=best_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
    crit = nn.CrossEntropyLoss()
    best_vl, best_sd, patience = float("inf"), None, 0
    for ep in range(100):
        model_final.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
            if ep < 10:
                for pg in opt.param_groups: pg["lr"] = best_lr * (ep + 1) / 10
            opt.zero_grad(); crit(model_final(xb), yb).backward(); opt.step()
        if ep >= 10: sched.step()
        if ep % 5 == 0:
            model_final.eval(); vl = 0.0
            with torch.no_grad():
                for xb, yb in train_dl:
                    xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                    vl += crit(model_final(xb), yb).item()
            vl /= len(train_dl)
            if vl < best_vl:
                best_vl = vl; best_sd = {k: v.cpu().clone() for k, v in model_final.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 3: break
    if best_sd is not None:
        model_final.load_state_dict(best_sd)

    # Per-drug threshold tuned on external val set
    proba_val = predict_proba(model_final, test_sets["Val"][0])
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(test_sets["Val"][1], proba_val >= t) for t in THRESHOLDS])]

    results = {}
    for name, (X_te, y_te) in test_sets.items():
        proba = predict_proba(model_final, X_te)
        preds = proba >= best_t
        results[name] = {
            "BalAcc": balanced_accuracy_score(y_te, preds),
            "AUC": roc_auc_score(y_te, proba),
            "Threshold": best_t,
            "Best_Param": f"lr={best_lr:.1e} drop={best_dh:.1f}",
        }
    return results

---
## Per-Drug Training Loop

In [ ]:
# ── 4. TRAIN LR + MLP PER DRUG ──

all_results = {}
for drug in tqdm(DRUGS_10, desc="LR+MLP"):
    print(f"\n{'='*60}")
    print(f"  {drug}")
    print(f"{'='*60}")

    X, y, species = load_drug_data(drug)
    if len(np.unique(y)) < 2:
        print(f"  SKIP: only one class present")
        continue

    # Species-stratified 70/15/15 split
    idx = np.arange(len(y)).reshape(-1, 1)
    idx_trval, idx_test, _, _ = stratified_species_drug_split(
        idx, y, species=species, test_size=0.15, random_state=SEED)
    idx_trval = idx_trval.flatten().astype(int)
    idx_test = idx_test.flatten().astype(int)

    val_frac = 0.15 / 0.85
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx_trval.reshape(-1, 1), y[idx_trval], species=species[idx_trval],
        test_size=val_frac, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)

    X_train_raw, y_train = X[idx_train], y[idx_train]
    X_val_raw,   y_val   = X[idx_val], y[idx_val]
    X_test_raw,  y_test  = X[idx_test], y[idx_test]

    # Preprocessing
    state = fit_input_transform(X_train_raw, "log1p+standardize")
    X_train_pp = apply_input_transform(X_train_raw, state)
    X_val_pp   = apply_input_transform(X_val_raw, state)
    X_test_pp  = apply_input_transform(X_test_raw, state)

    test_sets = {"Val": (X_val_pp, y_val), "Test": (X_test_pp, y_test)}
    drug_results = {}

    # LR
    print("  --- LR ---")
    drug_results["LR"] = train_eval_lr(X_train_pp, y_train, test_sets)
    print(f"    Val BalAcc={drug_results['LR']['Val']['BalAcc']:.4f}  Test BalAcc={drug_results['LR']['Test']['BalAcc']:.4f}")

    # MLP
    print("  --- MLP ---")
    drug_results["MLP"] = train_eval_mlp(X_train_pp, y_train, test_sets)
    print(f"    Val BalAcc={drug_results['MLP']['Val']['BalAcc']:.4f}  Test BalAcc={drug_results['MLP']['Test']['BalAcc']:.4f}")

    all_results[drug] = drug_results

print("\nLR + MLP complete.")

---
## Results: LR + MLP

In [ ]:
# ── 5. BUILD RESULT DATAFRAMES ──

ba_rows, auc_rows = [], []
for drug in DRUGS_10:
    if drug not in all_results: continue
    ba_row = {"Drug": drug}; auc_row = {"Drug": drug}
    for model in ["LR", "MLP"]:
        ba_row[model] = all_results[drug][model]["Test"]["BalAcc"]
        auc_row[model] = all_results[drug][model]["Test"]["AUC"]
    ba_rows.append(ba_row); auc_rows.append(auc_row)

df_ba = pd.DataFrame(ba_rows).set_index("Drug")
df_auc = pd.DataFrame(auc_rows).set_index("Drug")
short_names = {d: d[:15] for d in df_ba.index}

print("\nBalanced Accuracy (Test set):")
print(df_ba.round(4).to_string())
print("\nAUC-ROC (Test set):")
print(df_auc.round(4).to_string())

In [ ]:
# ── Heatmaps: LR + MLP ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3.5))

df_disp_ba = df_ba.copy(); df_disp_ba.index = [short_names[d] for d in df_disp_ba.index]
sns.heatmap(df_disp_ba.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy", "shrink": 0.8}, ax=ax1)
ax1.set_title("Balanced Accuracy (LR + MLP)", fontsize=12, fontweight="bold")

df_disp_auc = df_auc.copy(); df_disp_auc.index = [short_names[d] for d in df_disp_auc.index]
sns.heatmap(df_disp_auc.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC", "shrink": 0.8}, ax=ax2)
ax2.set_title("AUC-ROC (LR + MLP)", fontsize=12, fontweight="bold")

fig.suptitle("Dedicated Per-Drug Models -- Aggregated 70/15/15 Test", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_lr_mlp.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Bar chart: LR vs MLP ──
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(df_ba)); w = 0.3
ax.bar(x - w/2, df_ba["LR"], w, label="LR", color="#1f77b4", edgecolor="white")
ax.bar(x + w/2, df_ba["MLP"], w, label="MLP", color="#ff7f0e", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(short_names.values(), fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy"); ax.set_title("LR vs MLP (Aggregated Test)")
ax.legend(fontsize=10); ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.grid(True, ls='--', lw=0.5, color='gray', alpha=0.5, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / "barchart_lr_mlp.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Best hyperparameters ──
param_rows = []
for drug in DRUGS_10:
    if drug not in all_results: continue
    row = {"Drug": drug[:20]}
    for m in ["LR", "MLP"]:
        if m in all_results[drug]:
            row[m] = all_results[drug][m]["Test"]["Best_Param"]
    param_rows.append(row)
df_params = pd.DataFrame(param_rows).set_index("Drug")
print("\nBest Hyperparameters:")
print(df_params.to_string())

# ── Save results ──
df_ba.to_csv(OUT_DIR / "results_balacc.csv")
df_auc.to_csv(OUT_DIR / "results_auc.csv")
df_params.to_csv(OUT_DIR / "best_params.csv")

print("\nSaved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")

---
**Done.** Analysis complete.